# MOC Grid Scaling — Courant Wave-Speed Adjustment

Mirrors **`tests/test_grid_scaling_long_pipe.py`**: preview how each pipe's design wave speed
`a₀` is back-adjusted to `a_adj = L / (N·dt)` so **Cr = 1.0** before running a long transient.

Use **`get_grid_report(dt)`** instead of committing to a 10,000-step simulation first.

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jlillywh/RTHYM-MOC/main?labpath=validation%2Fnotebooks%2Fgrid_scaling_verification.ipynb)

## 1. Setup

In [ ]:
import rthym_moc as m
from rthym_moc.report import format_grid_report, summarize_grid_report
from _notebook_setup import bootstrap_validation_notebook

REPO_ROOT, TESTS_DIR, VALIDATION_DIR = bootstrap_validation_notebook()
print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc: {getattr(m, '__version__', 'unknown')}")

## 2. Build a long-pipe test case

In [ ]:
DT = 0.01
LENGTH_FT = 4000.0
SEGMENT_CAP = 50

solver = m.MOCSolver()
r1 = m.NodeInput()
r1.id, r1.type, r1.elevation, r1.head = "R1", "PressureBoundary", 100.0, 320.0
r2 = m.NodeInput()
r2.id, r2.type, r2.elevation, r2.head = "R2", "PressureBoundary", 100.0, 320.0
p1 = m.PipeInput()
p1.id, p1.from_node, p1.to_node = "P1", "R1", "R2"
p1.length, p1.diameter, p1.roughness, p1.flow_gpm = LENGTH_FT, 12.0, 130.0, 0.0
solver.add_node(r1)
solver.add_node(r2)
solver.add_pipe(p1)
solver.set_grid_policy(
    max_segments_per_pipe=SEGMENT_CAP,
    max_wave_speed_distortion=0.15,
    distortion_action="warn",
)

## 3. Preview grid (no time integration)

In [ ]:
report = solver.get_grid_report(DT)
print(format_grid_report(report))
table = summarize_grid_report(report)
row = table["P1"]
assert row["num_segments"] == SEGMENT_CAP
assert abs(row["courant_number"] - 1.0) < 1e-6
print("PASS: Courant number ≈ 1.0 and segment cap applied.")

## 4. Confirm run() metadata matches preview

In [ ]:
results = solver.run(total_time=0.02, dt=DT)
for key in (
    "pipe_wave_speed_design_fps",
    "pipe_wave_speed_adjusted_fps",
    "pipe_distortion_pct",
    "pipe_num_segments",
):
    assert results[key]["P1"] == report[key]["P1"], key
print("PASS: run() grid metadata matches get_grid_report().")